# 01_data_ingestion

This notebook handles API ingestion and CSV data loading. It includes robust error handling for API calls and converts date-like columns to `datetime64` during loading to simplify later analysis.

In [ ]:
import json
from pathlib import Path
import re

import pandas as pd
import requests

In [ ]:
RAW_DIR = Path("data/raw")
PROCESSED_DIR = Path("data/processed")
SCHEMES = {
    "125497": "HDFC_Top_100_Direct",
    "119551": "SBI_Bluechip",
    "120503": "ICICI_Bluechip",
    "118632": "Nippon_Large_Cap",
    "119092": "Axis_Bluechip",
    "120841": "Kotak_Bluechip",
}

def ensure_directories():
    for path in [RAW_DIR, PROCESSED_DIR, Path("notebooks"), Path("sql"), Path("dashboard"), Path("reports")]:
        path.mkdir(parents=True, exist_ok=True)

def infer_datetime_columns(df: pd.DataFrame) -> pd.DataFrame:
    date_candidates = [col for col in df.columns if re.search(r"date|day|month|year|period", col, re.I)]
    for col in date_candidates:
        if col not in df.columns:
            continue
        if pd.api.types.is_datetime64_any_dtype(df[col]):
            continue
        if not pd.api.types.is_string_dtype(df[col]) and not pd.api.types.is_object_dtype(df[col]):
            continue

        converted = pd.to_datetime(df[col], errors="coerce", infer_datetime_format=True, dayfirst=True)
        if converted.notna().sum() >= max(1, int(len(df) * 0.2)):
            df[col] = converted
            print(f"Converted {col} to datetime64[ns]")
    return df

def load_csv_datasets(raw_dir: Path) -> dict[str, pd.DataFrame]:
    csv_files = sorted(raw_dir.glob("*.csv"))
    datasets = {}
    if not csv_files:
        print("No CSV files found in data/raw/. Add your CSV files and rerun this notebook.")
        return datasets

    for csv_file in csv_files:
        try:
            df = pd.read_csv(csv_file, low_memory=False)
        except Exception as exc:
            print(f"Failed to read {csv_file.name}: {exc}")
            continue

        df = infer_datetime_columns(df)
        normalized_name = re.sub(r"^\d+_", "", csv_file.stem.lower().strip())
        print(
            f"Loaded {csv_file.name}: shape={df.shape}, normalized_name={normalized_name}"
        )
        datasets[normalized_name] = df
    return datasets

def safe_fetch_json(url: str) -> dict | None:
    try:
        response = requests.get(url, timeout=15)
        response.raise_for_status()
    except requests.exceptions.RequestException as exc:
        print(f"API request failed for {url}: {exc}")
        return None

    try:
        return response.json()
    except json.JSONDecodeError as exc:
        print(f"Failed to decode JSON from {url}: {exc}")
        return None

def nav_api_to_csv(scheme_code: str, scheme_name: str, raw_dir: Path) -> Path | None:
    url = f"https://api.mfapi.in/mf/{scheme_code}"
    print(f"Fetching NAV for {scheme_name} ({scheme_code}) from {url}")
    data = safe_fetch_json(url)
    if data is None:
        return None

    records = []
    for item in data.get("data", []):
        records.append({
            "scheme_code": scheme_code,
            "scheme_name": scheme_name,
            "date": item.get("date"),
            "nav": item.get("nav"),
        })

    if not records:
        print(f"No NAV records returned for {scheme_name}.")
        return None

    df = pd.DataFrame(records)
    df["date"] = pd.to_datetime(df["date"], errors="coerce", infer_datetime_format=True, dayfirst=True)

    out_path = raw_dir / f"nav_{scheme_code}_{scheme_name}.csv"
    df.to_csv(out_path, index=False)
    print(f"Saved NAV history to {out_path}")
    return out_path

def fetch_all_nav(raw_dir: Path):
    for scheme_code, scheme_name in SCHEMES.items():
        nav_api_to_csv(scheme_code, scheme_name, raw_dir)

In [ ]:
ensure_directories()
print("
Loading local CSV datasets from data/raw/ ...")
datasets = load_csv_datasets(RAW_DIR)

print("
Fetching NAV history from the MF API ...")
fetch_all_nav(RAW_DIR)

print("
Loaded dataset keys:")
print(sorted(datasets.keys()))